# Entraînement YOLOv8 — Passages piétons (DefisAccess)

**Avant de commencer :** Exécution → Modifier le type d'exécution → **GPU T4**

Exécutez les cellules dans l'ordre avec **Shift+Entrée**.

## Étape 1 — Vérifier le GPU

In [ ]:
!nvidia-smi

## Étape 2 — Installer les librairies

In [ ]:
!pip install ultralytics roboflow -q
import ultralytics
ultralytics.checks()

## Étape 3 — Télécharger le dataset depuis Roboflow

In [ ]:
import getpass
from roboflow import Roboflow

# Le champ ci-dessous masque votre clé comme un mot de passe
api_key = getpass.getpass('Clé API Roboflow : ')

rf = Roboflow(api_key=api_key)
project = rf.workspace('rey-kjtp1').project('Test_export_data_annotée')
dataset = project.version(1).download('yolov8')

print('Dataset dans :', dataset.location)

In [ ]:
import os
loc = dataset.location
print('data.yaml    :', os.path.exists(loc + '/data.yaml'))
print('Images train :', len(os.listdir(loc + '/train/images')), 'fichiers')
print('Images val   :', len(os.listdir(loc + '/valid/images')), 'fichiers')
print('Labels train :', len(os.listdir(loc + '/train/labels')), 'fichiers')

## Étape 4 — Entraînement YOLOv8

Durée estimée avec GPU T4 : **10 à 20 minutes**.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=dataset.location + '/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='passage_pieton_v2',
    patience=20,
    save=True,
    plots=True,
)

print('Entrainement termine !')
print('Dossier resultats :', results.save_dir)

## Étape 5 — Résultats

- **mAP50 > 0.5** = bon modèle
- **mAP50 > 0.7** = très bon modèle

In [ ]:
from ultralytics import YOLO
import glob

best_path = sorted(glob.glob('runs/detect/passage_pieton_v2*/weights/best.pt'))[-1]
print('Modele :', best_path)

model_eval = YOLO(best_path)
metrics = model_eval.val()

print('mAP50     :', round(metrics.box.map50, 3))
print('Precision :', round(metrics.box.mp, 3))
print('Rappel    :', round(metrics.box.mr, 3))

In [ ]:
import glob, os
from IPython.display import Image, display

run_dir = sorted(glob.glob('runs/detect/passage_pieton_v2*'))[-1]

for plot in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    path = run_dir + '/' + plot
    if os.path.exists(path):
        print('---', plot, '---')
        display(Image(path, width=800))

## Étape 6 — Sauvegarder best.pt

In [ ]:
import shutil, glob, os
from google.colab import drive, files

drive.mount('/content/drive')

best_path = sorted(glob.glob('runs/detect/passage_pieton_v2*/weights/best.pt'))[-1]

drive_dest = '/content/drive/MyDrive/DefiAccess2/testeIAPP/best.pt'
os.makedirs(os.path.dirname(drive_dest), exist_ok=True)
shutil.copy(best_path, drive_dest)
print('Sauvegarde sur Drive :', drive_dest)

files.download(best_path)
print('Telechargement lance !')

## Étape suivante (en local)

Copiez `best.pt` dans votre projet local :
```
Prototype_Appli_DefisAccess/models/best.pt
```
Le script `src/IA_PP.py` chargera automatiquement ce nouveau modèle.